# XAI Benchmark (MURA) — Kaggle runner

**Setup (once):**
1. Right sidebar → **Add data** → search **MURA** and add the MURA-v1.1 dataset.
2. Add the code: either set `CODE_SRC` to your GitHub repo URL below, **or** upload the
   `XAI_project` folder as a Kaggle dataset (Add data → your dataset).
3. Right sidebar → **Settings → Accelerator → GPU (T4/P100)**.
4. Run all cells. Results are written to `/kaggle/working/XAI_project/results/results.csv`
   (download from the **Output** tab).

In [ ]:
import glob, os, sys, shutil, subprocess, pathlib

# --- locate MURA-v1.1 under /kaggle/input ---
mura = glob.glob('/kaggle/input/**/MURA-v1.1', recursive=True)
assert mura, "Add the MURA dataset via 'Add data' (search MURA-v1.1)."
MURA_ROOT = mura[0]
assert os.path.exists(os.path.join(MURA_ROOT, 'train_image_paths.csv')), \
    f"train_image_paths.csv not found under {MURA_ROOT}"
print("MURA:", MURA_ROOT)

In [ ]:
# --- get the code (GitHub OR uploaded Kaggle dataset) ---
CODE_SRC = "https://github.com/CrusarAppy/mura-xai-benchmark.git"   # clones the pushed code
WORK = "/kaggle/working/XAI_project"

if CODE_SRC:
    if os.path.exists(WORK): shutil.rmtree(WORK)
    subprocess.run(["git", "clone", CODE_SRC, WORK], check=True)
else:
    found = glob.glob('/kaggle/input/**/scripts/run_experiment.py', recursive=True)
    assert found, "Set CODE_SRC to a GitHub URL, or upload the XAI_project folder as a Kaggle dataset."
    src_root = str(pathlib.Path(found[0]).parents[1])   # folder containing scripts/
    if os.path.exists(WORK): shutil.rmtree(WORK)
    shutil.copytree(src_root, WORK)
print("Code:", WORK)

In [ ]:
# --- install the package (Kaggle already ships torch + CUDA) ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", WORK], check=True)

In [ ]:
# --- patch the config: dataset path + quick debug first ---
import yaml
cfg_path = os.path.join(WORK, "configs", "densenet_gradcam.yaml")
cfg = yaml.safe_load(open(cfg_path))
cfg["data"]["mura_root"] = MURA_ROOT
cfg["train"]["quick_debug"] = True    # << set to False for the full run
yaml.safe_dump(cfg, open(cfg_path, "w"), sort_keys=False)
print(open(cfg_path).read())

In [ ]:
# --- device check ---
sys.path.insert(0, os.path.join(WORK, "src"))
from xai_bench.utils import get_device
print("Device:", get_device())

In [ ]:
# --- run the experiment ---
subprocess.run([sys.executable, os.path.join(WORK, "scripts", "run_experiment.py"),
                "--config", cfg_path], check=True, cwd=WORK)

In [ ]:
# --- view results ---
import pandas as pd
print(pd.read_csv(os.path.join(WORK, "results", "results.csv")).T)

### Full run
Re-run the config cell with `cfg["train"]["quick_debug"] = False`, then re-run the experiment cell.
To sweep folds/seeds, change `cfg["data"]["fold"]` / `cfg["experiment"]["seed"]` and re-run.

## Full sweep — all 18 configs (3 backbones x 6 XAI methods)

Trains each backbone once, reuses it for all 6 explainers, writes every row to
`results/sweep_results.csv`. Smoke-test with `quick_debug=True` first, then flip to `False`.

In [ ]:
# --- patch the sweep config: dataset path + quick_debug smoke test first ---
import yaml
sweep_path = os.path.join(WORK, "configs", "sweep.yaml")
scfg = yaml.safe_load(open(sweep_path))
scfg["data"]["mura_root"] = MURA_ROOT
scfg["train"]["quick_debug"] = True     # << set False for the real multi-hour run
yaml.safe_dump(scfg, open(sweep_path, "w"), sort_keys=False)
print(open(sweep_path).read())

In [ ]:
# --- run the sweep (optionally a subset of folds this session) ---
# Kaggle sessions cap at ~12h. 5 folds x 3 backbones may not fit in one commit,
# so run a batch of folds per session, download results/sweep_results.csv each time,
# then combine locally with scripts/aggregate_cis.py.
FOLDS_THIS_SESSION = "0,1,2"   # e.g. "0,1,2" then "3,4" in a second commit; "" = use config
cmd = [sys.executable, os.path.join(WORK, "scripts", "run_sweep.py"), "--config", sweep_path]
if FOLDS_THIS_SESSION.strip():
    cmd += ["--folds", FOLDS_THIS_SESSION]
subprocess.run(cmd, check=True, cwd=WORK)

In [ ]:
# --- view sweep results (18 rows) ---
import pandas as pd
df = pd.read_csv(os.path.join(WORK, "results", "sweep_results.csv"))
print(f"{len(df)} rows")
print(df[["backbone","xai_method","auroc","ece","deletion_auc","insertion_auc",
          "runtime_s_per_explanation"]].to_string(index=False))

### Confidence intervals
After all folds are collected into `results/sweep_results.csv` (combine multiple
session CSVs locally if you batched), compute mean +/- 95% CI per (backbone, method).

In [ ]:
# --- aggregate to mean +/- 95% CI ---
subprocess.run([sys.executable, os.path.join(WORK, "scripts", "aggregate_cis.py"),
                os.path.join(WORK, "results", "sweep_results.csv"),
                "--out", os.path.join(WORK, "results", "summary_cis.csv")],
               check=True, cwd=WORK)